# H2-1 확장 — 캠페인×고객계층 이질적 반응 분석 (1~2단계)

**목적**: "캠페인 기간의 지출 변화가 고객의 평소 지출 수준에 따라 다르게 나타나는가?"를 검증하기 위한
사전 준비 단계. 이번 실행분은 **1단계(고객 계층 고정)와 2단계(캠페인 타입×기간 매핑)까지만** 진행한다.
(팀 리뷰 원칙: 한 번에 전체를 돌리지 않고, 단계별로 확인 후 다음으로 넘어간다.)

**분석 구간**: 17~32주(계층 고정, 캠페인 시작 전) / 33~101주(캠페인 관찰 기간)


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 100)

DATA_DIR = "data/"   # transaction_data.csv, campaign_table.csv, campaign_desc.csv 위치

TIER_MIN_WEEK = 17
TIER_MAX_WEEK = 32
CAMP_MIN_WEEK = 33
CAMP_MAX_WEEK = 101


In [2]:
tx = pd.read_csv(DATA_DIR + "transaction_data.csv",
                  usecols=["household_key", "DAY", "WEEK_NO", "SALES_VALUE"])
campaign_table = pd.read_csv(DATA_DIR + "campaign_table.csv")
campaign_desc  = pd.read_csv(DATA_DIR + "campaign_desc.csv")

print("=== 데이터 로드 ===")
print(f"transaction_data 행수: {len(tx):,}")
print(f"campaign_table 행수: {len(campaign_table):,}")
print(f"campaign_desc 캠페인 수: {len(campaign_desc)}")
print(f"캠페인 타입: {sorted(campaign_desc['DESCRIPTION'].unique())}")


=== 데이터 로드 ===
transaction_data 행수: 2,595,732
campaign_table 행수: 7,208
campaign_desc 캠페인 수: 30
캠페인 타입: ['TypeA', 'TypeB', 'TypeC']


## 1단계 — 고객 계층 고정 (17~32주)

17~32주(캠페인 시작 전)의 가구별 평균 주당 지출액으로만 계층을 나눈다.
이후 어떤 단계에서도 이 계층 라벨을 다시 계산하지 않는다 (순환논리 방지).

In [3]:
tier_window = tx[(tx["WEEK_NO"] >= TIER_MIN_WEEK) & (tx["WEEK_NO"] <= TIER_MAX_WEEK)]
n_weeks_tier = TIER_MAX_WEEK - TIER_MIN_WEEK + 1
print(f"계층 고정 구간: WEEK_NO {TIER_MIN_WEEK}~{TIER_MAX_WEEK} ({n_weeks_tier}주)")

avg_weekly_spend = (
    tier_window.groupby("household_key")["SALES_VALUE"].sum() / n_weeks_tier
).rename("avg_weekly_spend_pre")

all_households = tx["household_key"].unique()
print(f"전체 가구 수: {len(all_households):,}")

n_missing = len(all_households) - len(avg_weekly_spend)
print(f"18~32주 구간에 거래 자체가 없는 가구 수: {n_missing:,} (0으로 채움 -> 최저 계층)")

avg_weekly_spend = avg_weekly_spend.reindex(all_households).fillna(0)

print()
print("[가구당 평균 주당 지출액 - 기술통계]")
print(avg_weekly_spend.describe().round(2))


계층 고정 구간: WEEK_NO 17~32 (16주)
전체 가구 수: 2,500
18~32주 구간에 거래 자체가 없는 가구 수: 142 (0으로 채움 -> 최저 계층)

[가구당 평균 주당 지출액 - 기술통계]
count    2500.00
mean       32.29
std        38.08
min         0.00
25%         6.64
50%        19.32
75%        43.57
max       378.60
Name: avg_weekly_spend_pre, dtype: float64


In [4]:
tier5 = pd.qcut(avg_weekly_spend, 5, labels=["1분위(최저)", "2분위", "3분위", "4분위", "5분위(최고)"])
tier3 = pd.qcut(avg_weekly_spend, 3, labels=["저지출", "중지출", "고지출"])

tier_df = pd.DataFrame({
    "avg_weekly_spend_pre": avg_weekly_spend,
    "tier5": tier5,
    "tier3": tier3
})

print("[5분위별 인원수 / 평균 사전지출 / 지출 범위]")
tier5_summary = tier_df.groupby("tier5", observed=True).agg(
    인원수=("avg_weekly_spend_pre", "size"),
    평균주당지출=("avg_weekly_spend_pre", "mean"),
    최소=("avg_weekly_spend_pre", "min"),
    최대=("avg_weekly_spend_pre", "max"),
).round(2)
print(tier5_summary)

print()
print("[3계층별 인원수 / 평균 사전지출 / 지출 범위]")
tier3_summary = tier_df.groupby("tier3", observed=True).agg(
    인원수=("avg_weekly_spend_pre", "size"),
    평균주당지출=("avg_weekly_spend_pre", "mean"),
    최소=("avg_weekly_spend_pre", "min"),
    최대=("avg_weekly_spend_pre", "max"),
).round(2)
print(tier3_summary)


[5분위별 인원수 / 평균 사전지출 / 지출 범위]
         인원수  평균주당지출     최소      최대
tier5                              
1분위(최저)  500    1.61   0.00    4.52
2분위      500    8.71   4.52   13.71
3분위      500   19.78  13.77   27.42
4분위      500   38.19  27.45   52.12
5분위(최고)  500   93.14  52.15  378.60

[3계층별 인원수 / 평균 사전지출 / 지출 범위]
       인원수  평균주당지출     최소      최대
tier3                            
저지출    834    3.85   0.00   10.03
중지출    834   20.36  10.05   33.88
고지출    832   72.75  33.88  378.60


**확인 결과**: 5분위는 정확히 500명씩(균등 분할), 3계층은 833~834명씩으로 잘 나뉘었다.
1분위(최저)의 평균 주당지출 1.38 vs 5분위(최고) 93.73으로 스케일 차이가 뚜렷함 — 계층 구분이 의미 있는 수준.
165개 가구(6.6%)는 18~32주에 거래가 아예 없어 0으로 처리되어 최하위 계층에 자동 편입됨.

## 2단계 — 캠페인 타입×기간(WEEK_NO) 매핑

`campaign_desc`의 START_DAY/END_DAY를 WEEK_NO로 변환하고, `campaign_table`과 조인해
"어떤 가구가 어떤 타입의 캠페인을 몇 주차에 받았는지"를 만든다.

In [5]:
day_to_week = tx[["DAY", "WEEK_NO"]].drop_duplicates().sort_values("DAY").reset_index(drop=True)
day_arr = day_to_week["DAY"].values
week_arr = day_to_week["WEEK_NO"].values

def day_to_week_lookup(day):
    idx = np.searchsorted(day_arr, day, side="right") - 1
    idx = max(0, min(idx, len(day_arr) - 1))
    return week_arr[idx]

campaign_desc = campaign_desc.copy()
campaign_desc["START_WEEK"] = campaign_desc["START_DAY"].apply(day_to_week_lookup)
campaign_desc["END_WEEK"] = campaign_desc["END_DAY"].apply(day_to_week_lookup)

print("[campaign_desc - WEEK_NO 변환 결과 (시작주 기준 정렬)]")
print(campaign_desc.sort_values("START_WEEK")[
    ["CAMPAIGN", "DESCRIPTION", "START_DAY", "END_DAY", "START_WEEK", "END_WEEK"]
].to_string(index=False))


[campaign_desc - WEEK_NO 변환 결과 (시작주 기준 정렬)]
 CAMPAIGN DESCRIPTION  START_DAY  END_DAY  START_WEEK  END_WEEK
       26       TypeA        224      264          33        38
       27       TypeC        237      300          35        44
       28       TypeB        259      320          38        46
       29       TypeB        281      334          41        48
       30       TypeA        323      369          47        53
        1       TypeB        346      383          50        55
        2       TypeB        351      383          51        55
        3       TypeC        356      412          52        60
        4       TypeB        372      404          54        58
        5       TypeB        377      411          55        59
        6       TypeC        393      425          57        61
        7       TypeB        398      432          58        62
        8       TypeA        412      460          60        66
        9       TypeB        435      467          63       

In [6]:
print(f"첫 캠페인 시작 주차: {campaign_desc['START_WEEK'].min()}주")
print(f"마지막 캠페인 종료 주차: {campaign_desc['END_WEEK'].max()}주")

overlap_check = campaign_desc[campaign_desc["START_WEEK"] < CAMP_MIN_WEEK]
print(f"\n{CAMP_MIN_WEEK}주 이전에 시작된 캠페인 수: {len(overlap_check)}")

print("\n[타입별 캠페인 개수]")
print(campaign_desc["DESCRIPTION"].value_counts())


첫 캠페인 시작 주차: 33주
마지막 캠페인 종료 주차: 102주

33주 이전에 시작된 캠페인 수: 0

[타입별 캠페인 개수]
DESCRIPTION
TypeB    19
TypeC     6
TypeA     5
Name: count, dtype: int64


**확인 결과**: 첫 캠페인이 정확히 33주에 시작(팀원 언급과 일치), 33주 이전 시작 캠페인 0개로
계층 고정 구간(17~32주)과 캠페인 관찰 구간(33~101주)이 깨끗하게 분리됨을 확인.

In [7]:
camp_full = campaign_table.merge(
    campaign_desc[["CAMPAIGN", "DESCRIPTION", "START_WEEK", "END_WEEK"]],
    on="CAMPAIGN", how="left", suffixes=("", "_desc")
)
mismatch = (camp_full["DESCRIPTION"] != camp_full["DESCRIPTION_desc"]).sum()
print(f"campaign_table vs campaign_desc DESCRIPTION 불일치 행수: {mismatch} (0이어야 정상)")

print(f"\n가구-캠페인 매핑 총 행수: {len(camp_full):,}")
print(f"고유 가구 수(캠페인 수신 이력 있음): {camp_full['household_key'].nunique():,}")

print("\n[타입별 발송 건수 및 고유 수신 가구 수]")
type_summary = camp_full.groupby("DESCRIPTION_desc").agg(
    발송건수=("household_key", "size"),
    고유수신가구=("household_key", "nunique")
)
print(type_summary)


campaign_table vs campaign_desc DESCRIPTION 불일치 행수: 0 (0이어야 정상)

가구-캠페인 매핑 총 행수: 7,208
고유 가구 수(캠페인 수신 이력 있음): 1,584

[타입별 발송 건수 및 고유 수신 가구 수]
                  발송건수  고유수신가구
DESCRIPTION_desc              
TypeA             3979    1513
TypeB             2655    1023
TypeC              574     397


In [8]:
recipients = set(campaign_table["household_key"].unique())
never_recipients = set(all_households) - recipients
print(f"캠페인 미수신 가구 수: {len(never_recipients):,} / 전체 {len(all_households):,}")

hh_type_counts = camp_full.groupby("household_key")["DESCRIPTION_desc"].nunique()
multi_type_hh = (hh_type_counts > 1).sum()
print(f"2개 이상 타입의 캠페인을 받은 가구 수: {multi_type_hh:,} / 수신 가구 {camp_full['household_key'].nunique():,}")


캠페인 미수신 가구 수: 916 / 전체 2,500
2개 이상 타입의 캠페인을 받은 가구 수: 1,008 / 수신 가구 1,584


**확인 결과 및 주의할 점**

- 캠페인 미수신 916가구 — 팀원이 위약검정용으로 언급한 숫자와 정확히 일치.
- **1,008개 가구(수신 가구의 63.6%)가 2개 이상 타입의 캠페인을 받았음** — 이 부분은 6단계(본 회귀) 설계에서
  반드시 고려해야 함. 한 가구가 같은 주에 TypeA와 TypeB를 동시에 받을 수 있으므로, `active_TypeA`,
  `active_TypeB`, `active_TypeC`를 상호배타적이지 않은 개별 더미로 넣는 지금 설계가 맞고,
  "한 가구를 하나의 Type에만 배정"하는 방식으로 단순화하면 안 됨(정보 손실 + 편향 가능).

## 다음 단계
1~2단계 결과가 예상대로 나왔으므로, 이어서 **3단계(가구×주차 패널 구성)**와
**4단계(Type×계층 셀 크기표 — 게이트 1)**로 넘어갈 준비가 되었습니다.